# Chi-outline

Implementation of the **chi-outline** algorithm from:

> Zhong, X., & Duckham, M. (2016). Characterizing the shapes of noisy, non-uniform, and disconnected point clusters in the plane. *Computers, Environment and Urban Systems*, 57, 48-58.

Chi-outline extends chi-shape (see `chi_shape.ipynb`) so that the output can be a set of **disconnected** simple polygons, and can **exclude outliers**, while still using a single parameter `chi`. It does this by relaxing chi-shape's single-polygon regularity constraint (allowing "isolating" exterior edges to be removed, splitting the shape), and by giving each resulting connected component its own adaptive length threshold based on the local statistics of its edge lengths.

The `ChiOutline` class is exposed by this package's C++/pybind11 extension:
```python
from concavehull import ChiOutline
co = ChiOutline(pts, chi=0.5)
polygons = co.compute()  # list of Nx2 arrays, one per disconnected region
```
Every numeric parameter below is an interactive slider -- drag them to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
from ipywidgets import interact, IntSlider, FloatSlider

from concavehull import concavehull, ChiOutline


def letter_points(letter, n_points, font_size=200, seed=None):
    """Sample `n_points` random points inside the filled outline of `letter`.

    Mirrors the semi-random letter-shaped point sets used throughout both
    papers (e.g. Duckham et al. 2008, Fig. 6/9; Zhong & Duckham 2016, Fig. 3),
    minus the minimum-pairwise-distance constraint they use -- plain uniform
    rejection sampling inside the glyph outline is good enough to show the
    algorithms' behaviour.
    """
    rng = np.random.default_rng(seed)
    prop = FontProperties(family="DejaVu Sans", weight="bold")
    path = TextPath((0, 0), letter, size=font_size, prop=prop)
    (xmin, ymin), (xmax, ymax) = path.get_extents().get_points()
    pts = np.empty((0, 2))
    while len(pts) < n_points:
        batch = rng.uniform([xmin, ymin], [xmax, ymax], size=(4 * n_points, 2))
        inside = path.contains_points(batch)
        pts = np.vstack([pts, batch[inside]])
    pts = pts[:n_points]
    pts = pts - pts.mean(axis=0)
    return pts


def normalize_scale(pts, target_width):
    """Rescale `pts` (already centered) so its widest axis spans `target_width`."""
    extent = np.ptp(pts, axis=0).max()
    return pts * (target_width / extent)


def plot_points_and_polygons(ax, pts, polygons, title=None, point_size=8):
    ax.scatter(pts[:, 0], pts[:, 1], s=point_size, color="tab:blue", alpha=0.5)
    for poly in polygons:
        closed = np.vstack([poly, poly[:1]])
        ax.plot(closed[:, 0], closed[:, 1], color="tab:red", linewidth=1.5)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title)

## Example 1: Motivation -- two clusters and outliers

This is the scenario chi-shape cannot handle: two separated clusters plus uniformly scattered outliers. chi-shape (left) is forced into a single polygon that bridges both clusters and still contains every outlier. chi-outline (right) correctly splits the shape into two disconnected polygons and excludes the outliers.

In [ ]:
@interact(
    n_per_cluster=IntSlider(value=80, min=20, max=300, step=10, description="n_per_cluster",
                             continuous_update=False),
    separation=FloatSlider(value=8.0, min=2.0, max=15.0, step=0.5, description="separation",
                            continuous_update=False),
    n_outliers=IntSlider(value=15, min=0, max=100, step=5, description="n_outliers",
                          continuous_update=False),
    chi_factor=FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description="chi_factor (left)",
                            continuous_update=False),
    chi=FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1, description="chi (right)",
                     continuous_update=False),
)
def show_motivation(n_per_cluster, separation, n_outliers, chi_factor, chi):
    rng = np.random.default_rng(0)
    cluster_a = rng.normal(loc=[0, 0], scale=0.3, size=(n_per_cluster, 2))
    cluster_b = rng.normal(loc=[separation, 0], scale=0.3, size=(n_per_cluster, 2))
    if n_outliers:
        outliers = rng.uniform(low=[-separation / 2, -separation / 2],
                                high=[separation * 1.5, separation / 2], size=(n_outliers, 2))
        pts = np.vstack([cluster_a, cluster_b, outliers])
    else:
        pts = np.vstack([cluster_a, cluster_b])

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    hull = concavehull(pts, chi_factor=chi_factor)
    plot_points_and_polygons(
        axes[0], pts, [hull],
        title="chi-shape\n(one polygon, bridges both clusters, keeps outliers)",
    )

    co = ChiOutline(pts, chi=chi)
    polys = co.compute()
    plot_points_and_polygons(
        axes[1], pts, polys,
        title=f"chi-outline\n({len(polys)} disconnected polygons, outliers excluded)",
    )

    fig.suptitle("Why chi-outline: chi-shape cannot separate clusters or drop outliers")
    plt.tight_layout()
    plt.show()

## Example 2: Letter grid (the paper's flagship experiment)

Section 4.1 of the paper places six hole-free letters -- "H", "E", "X", "C", "S", "W" -- on a 2x3 grid, far enough apart that the expected footprint is disconnected, then adds random outliers drawn from the surrounding area (Fig. 3a). We reproduce that setup directly; `pts_grid` is kept around for Example 3 below.

In [ ]:
letters = ["H", "E", "X", "C", "S", "W"]
grid_cols = 3


def build_pts_grid(n_points_per_letter, grid_spacing, letter_width, outlier_fraction, seed=42):
    rng = np.random.default_rng(seed)
    cluster_pts = []
    for i, letter in enumerate(letters):
        row, col = divmod(i, grid_cols)
        offset = np.array([col * grid_spacing, -row * grid_spacing])
        pts_i = normalize_scale(letter_points(letter, n_points_per_letter, seed=1000 + i), letter_width)
        cluster_pts.append(pts_i + offset)
    cluster_pts = np.vstack(cluster_pts)

    xmin, ymin = cluster_pts.min(axis=0) - 1
    xmax, ymax = cluster_pts.max(axis=0) + 1
    n_outliers = int(outlier_fraction * len(cluster_pts))
    if n_outliers:
        outliers = rng.uniform([xmin, ymin], [xmax, ymax], size=(n_outliers, 2))
        return np.vstack([cluster_pts, outliers])
    return cluster_pts


pts_grid = build_pts_grid(90, 6.0, 3.5, 0.15)  # default; Example 3 reuses this


@interact(
    n_points_per_letter=IntSlider(value=90, min=20, max=300, step=10, description="pts/letter",
                                   continuous_update=False),
    grid_spacing=FloatSlider(value=6.0, min=3.0, max=12.0, step=0.5, description="grid_spacing",
                              continuous_update=False),
    letter_width=FloatSlider(value=3.5, min=1.5, max=5.5, step=0.25, description="letter_width",
                              continuous_update=False),
    outlier_fraction=FloatSlider(value=0.15, min=0.0, max=0.6, step=0.05, description="outlier_frac",
                                  continuous_update=False),
    chi=FloatSlider(value=1.3, min=0.1, max=3.0, step=0.05, description="chi",
                     continuous_update=False),
)
def show_letter_grid(n_points_per_letter, grid_spacing, letter_width, outlier_fraction, chi):
    global pts_grid
    pts_grid = build_pts_grid(n_points_per_letter, grid_spacing, letter_width, outlier_fraction)

    co = ChiOutline(pts_grid, chi=chi)
    polys_grid = co.compute()

    fig, ax = plt.subplots(figsize=(10, 7))
    plot_points_and_polygons(
        ax, pts_grid, polys_grid,
        title=f"chi-outline on a 2x3 letter grid + {outlier_fraction:.0%} outliers "
              f"({len(polys_grid)} polygons found)",
    )
    plt.show()

## Example 3: Effect of the `chi` parameter

`chi` trades off the same two failure modes the paper discusses in Section 6: too low and the per-component threshold gets aggressive, fragmenting real letters (and keeping small noise clusters); too high and it stops carving, eventually merging letters and outliers into one blob. Drag the slider over the letter-grid example (`pts_grid`, from Example 2 above) to find the sweet spot (around `chi=1.0-1.3` for the default grid parameters).

In [ ]:
@interact(chi=FloatSlider(value=1.3, min=0.1, max=3.0, step=0.05, description="chi",
                           continuous_update=False))
def show_chi_outline(chi):
    co = ChiOutline(pts_grid, chi=chi)
    polys = co.compute()
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_points_and_polygons(ax, pts_grid, polys, title=f"chi={chi:.2f} -> {len(polys)} polygons")
    plt.show()

## Example 4: Non-uniform cluster density

Experiment 3 of the paper (Fig. 6-8) specifically tests clusters with *different* point densities, which is where chi-outline's adaptive per-connected-component threshold (Eqs. 3-4) is designed to help over a single global parameter. Here three clusters of independently adjustable point counts (and therefore densities, at fixed spatial scale), plus outliers, are all recovered in one run.

In [ ]:
@interact(
    dense_n=IntSlider(value=300, min=50, max=600, step=25, description="dense_n",
                       continuous_update=False),
    medium_n=IntSlider(value=100, min=20, max=400, step=10, description="medium_n",
                        continuous_update=False),
    sparse_n=IntSlider(value=30, min=5, max=200, step=5, description="sparse_n",
                        continuous_update=False),
    n_outliers=IntSlider(value=20, min=0, max=100, step=5, description="n_outliers",
                          continuous_update=False),
    chi=FloatSlider(value=0.6, min=0.1, max=2.5, step=0.05, description="chi",
                     continuous_update=False),
)
def show_density(dense_n, medium_n, sparse_n, n_outliers, chi):
    rng = np.random.default_rng(7)
    dense_cluster = rng.normal(loc=[0, 0], scale=0.3, size=(dense_n, 2))
    medium_cluster = rng.normal(loc=[6, 0], scale=0.3, size=(medium_n, 2))
    sparse_cluster = rng.normal(loc=[3, 6], scale=0.3, size=(sparse_n, 2))
    clusters = [dense_cluster, medium_cluster, sparse_cluster]
    if n_outliers:
        outliers = rng.uniform(low=[-3, -3], high=[9, 9], size=(n_outliers, 2))
        clusters.append(outliers)
    pts_density = np.vstack(clusters)

    co = ChiOutline(pts_density, chi=chi)
    polys_density = co.compute()

    fig, ax = plt.subplots(figsize=(7, 7))
    plot_points_and_polygons(
        ax, pts_density, polys_density,
        title=(
            f"Clusters with different densities ({dense_n}/{medium_n}/{sparse_n} pts) + outliers\n"
            f"-> {len(polys_density)} polygons via adaptive per-component threshold"
        ),
    )
    plt.show()

## Example 5: Noise-to-signal ratio (NSR)

Experiment 1 of the paper sweeps the proportion of outliers to signal points (NSR) from 0% to 50%. We reproduce the same sweep on a two-cluster point set with adjustable cluster size, separation, and chi.

In [ ]:
@interact(
    n_per_cluster=IntSlider(value=100, min=20, max=300, step=10, description="n_per_cluster",
                             continuous_update=False),
    separation=FloatSlider(value=8.0, min=2.0, max=15.0, step=0.5, description="separation",
                            continuous_update=False),
    chi=FloatSlider(value=0.6, min=0.1, max=2.5, step=0.05, description="chi",
                     continuous_update=False),
)
def show_nsr_sweep(n_per_cluster, separation, chi):
    rng = np.random.default_rng(3)
    cluster_a = rng.normal(loc=[0, 0], scale=0.3, size=(n_per_cluster, 2))
    cluster_b = rng.normal(loc=[separation, 0], scale=0.3, size=(n_per_cluster, 2))
    n_signal = len(cluster_a) + len(cluster_b)

    nsrs = [0.0, 0.2, 0.4, 0.6]
    fig, axes = plt.subplots(1, len(nsrs), figsize=(4 * len(nsrs), 4))
    for nsr, ax in zip(nsrs, axes):
        n_out = int(nsr * n_signal)
        if n_out:
            outliers = rng.uniform(low=[-separation / 2, -separation / 2],
                                    high=[separation * 1.5, separation / 2], size=(n_out, 2))
            pts_nsr = np.vstack([cluster_a, cluster_b, outliers])
        else:
            pts_nsr = np.vstack([cluster_a, cluster_b])
        co = ChiOutline(pts_nsr, chi=chi)
        polys_nsr = co.compute()
        plot_points_and_polygons(ax, pts_nsr, polys_nsr, title=f"NSR={nsr:.0%}\n{len(polys_nsr)} polygons")
    fig.suptitle(f"Effect of noise-to-signal ratio (NSR), chi={chi:.2f}, cf. Zhong & Duckham 2016, Experiment 1")
    plt.tight_layout()
    plt.show()

## Notes

- This implementation recomputes vertex degrees and connected components with direct scans of the triangulation rather than the paper's incremental bookkeeping, so it is worst-case `O(n^2)` rather than `O(n log n)` -- fine at the point-cloud sizes used above, worth knowing before throwing very large inputs at it.
- See `chi_shape.ipynb` for the simpler, single-polygon chi-shape algorithm this one extends, and `bottleneck_split.ipynb` for splitting a single merged polygon (from either algorithm) into its constituent objects.